# RQ1 — effort-reduction curve (TESSERA, 0%–100% expert)

**Purpose.** Builds the full additive effort-reduction curve
(`docs/research_questions.md#rq1--foundation-models-vs-conventional-approaches`):
citizen-science presences as a fixed base, expert polygons added on top
in increasing amounts, to find where added field-collection effort
stops paying for itself. The two endpoints (0% = citizen-science only,
100% = expert only, no citizen science — kept as its own case per the
adopted design, not "citizen science + all expert") were already run
separately in
[`one_class_method_selection.ipynb`](one_class_method_selection.ipynb) /
[`presence_background_citizen_science.ipynb`](presence_background_citizen_science.ipynb)
and
[`expert_arm_classifiers.ipynb`](expert_arm_classifiers.ipynb); this
notebook recomputes **all 11 points (0%, 10%, …, 100%) under one
unified, fixed-test-set methodology**, so the whole curve is
comparable — the earlier two notebooks each drew their own independent
test-block split, which is fine standalone but not valid for reading a
trend across points.

**Design decisions made for this notebook (mine, not yet confirmed by
the supervisor — proposed to Íris and approved 2026-11-06):**

1. **Mixing unit: polygons, not pixels.** "X% expert" = a random X% of
   the expert *polygons* (every pixel inside them), not X% of pixels
   scattered across all polygons — collecting expert data costs a
   field visit per site/patch, not per pixel, so this is the effort
   unit that actually matches "cost of going to the field."
2. **Nested sampling.** Which polygons count as "effort already spent"
   is fixed by one random order (seed 0) for the whole curve, so 20%'s
   polygons are a superset of 10%'s — the curve reads as steadily
   adding effort, not redrawing a fresh random sample at every point.
3. **One fixed test set per evaluation seed, reused at every mixing
   level.** Test blocks are drawn once per seed (not once per level)
   from the 10 km blocks that contain citizen-science or expert
   presences, and held constant across all 11 levels within that seed
   — otherwise a score change across the curve could just mean
   "different test data," not "more expert data helped." Still
   averaged over 5 seeds for robustness, as in the two endpoint
   notebooks.
4. **The test presence set always includes both sources**, even at
   100% (expert-only training) — this also shows whether a model
   trained purely on expensive data generalises back to the
   citizen-science distribution, not just to more expert data.

Runs all three classifiers already chosen/tested: **One-Class SVM**
(`nu=0.05`), **Random Forest** (primary presence-vs-background) and
**Logistic Regression** (reported baseline), same background pool and
metrics (ROC-AUC, continuous Boyce) as every other RQ1 notebook so
far.


## 1. Load pilot embeddings, background pool, citizen-science and expert presences

Expert presence pixels are recovered the same way as
`expert_arm_classifiers.ipynb` (spatial join against
`expert_polygons_clean.gpkg`, since there is no precomputed `pixel`
column for polygons), but this time also keeping, for every matched
pixel, **which polygon it came from** (`index_right` from the spatial
join) — needed for the polygon-level mixing in Section 2. The one
known overlapping polygon pair (1.62 m², see `expert_polygons.ipynb`)
can match a pixel to two polygons; duplicates are dropped, keeping the
first match, so every pixel counts once.


In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from src.embeddings import block_id, load_pilot_embeddings, match_to_pixel

EMB_DIR = ROOT / "data" / "processed" / "embeddings"
PILOT_DIR = EMB_DIR / "tessera_v1.1_2023_rq1_pilot"
BG_FILE = EMB_DIR / "tessera_v1.1_2023_rq1_background" / "background_points.npz"

if not BG_FILE.exists():
    raise FileNotFoundError(
        f"{BG_FILE} is missing — run one_class_method_selection.ipynb first, "
        "it builds the shared background pool this notebook reuses."
    )

pilot_df, pilot_emb = load_pilot_embeddings(PILOT_DIR)
bg = np.load(BG_FILE)
bg_x, bg_y, bg_emb = bg["x"], bg["y"], bg["emb"]

# citizen-science presences
dealbata = pd.read_csv(ROOT / "data" / "interim" / "citizen_science" / "inat_dealbata_clean.csv")
cs_x, cs_y, cs_emb, cs_n_matched = match_to_pixel(dealbata["pixel"], pilot_df, pilot_emb)

# expert presences, pixel -> polygon id
expert_gdf = gpd.read_file(ROOT / "data" / "interim" / "expert_reference" / "expert_polygons_clean.gpkg").to_crs(32629)
pilot_points = gpd.GeoDataFrame(pilot_df, geometry=gpd.points_from_xy(pilot_df["x"], pilot_df["y"]), crs=32629)
joined = gpd.sjoin(pilot_points, expert_gdf[["geometry"]], predicate="within", how="inner")
joined = joined[~joined.index.duplicated(keep="first")]
expert_idx = joined.index.values
expert_poly_id_all = joined["index_right"].values
expert_emb_all = pilot_emb[expert_idx]
valid = ~np.isnan(expert_emb_all).any(axis=1)
expert_x, expert_y = pilot_df["x"].values[expert_idx][valid], pilot_df["y"].values[expert_idx][valid]
expert_emb = expert_emb_all[valid]
expert_poly_id = expert_poly_id_all[valid]

cs_block = block_id(cs_x, cs_y)
expert_block = block_id(expert_x, expert_y)
bg_block = block_id(bg_x, bg_y)

print(f"citizen-science presences: {len(cs_emb):,} (matched {cs_n_matched}/{len(dealbata)})")
print(f"expert presence pixels: {len(expert_emb):,}, across {len(np.unique(expert_poly_id)):,} polygons")
print(f"background points: {len(bg_emb):,}")


## 2. Effort order and mixing levels

A fixed random order over the expert polygons (seed 0, independent of
the 5 evaluation seeds in Section 3) decides which polygons "count" as
effort already spent at each level — this order is reused for every
evaluation seed, so the x-axis always means the same subset of
polygons, only the test/train split underneath it changes. Reported
`total_N` is nominal (full dataset, not one train fold) — the citizen-
science base (fixed) plus the expert pixels belonging to the included
polygons at that level, matching `docs/research_questions.md`'s
"report total N alongside the mixing percentage."


In [ ]:
LEVELS = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

poly_ids_available = np.unique(expert_poly_id)
poly_order = np.random.default_rng(0).permutation(poly_ids_available)

def included_polygons(level_pct):
    if level_pct >= 100:
        return poly_ids_available
    n_incl = round(level_pct / 100 * len(poly_order))
    return poly_order[:n_incl]

nominal_rows = []
for lvl in LEVELS:
    incl = included_polygons(lvl)
    n_expert_px = len(expert_poly_id) if lvl >= 100 else int(np.isin(expert_poly_id, incl).sum())
    n_cs = 0 if lvl >= 100 else len(cs_emb)
    nominal_rows.append({
        "level_pct": lvl, "n_polygons_included": len(incl),
        "n_expert_pixels": n_expert_px, "n_cs_presences": n_cs, "total_N": n_cs + n_expert_px,
    })
nominal_df = pd.DataFrame(nominal_rows)
nominal_df


## 3. Fixed spatial-block test set per seed

Test blocks are drawn once per seed from the blocks containing either
citizen-science or expert presences, and reused at every mixing level
within that seed (see "Design decisions" above). 20% of candidate
blocks held out, 5 seeds.


In [ ]:
candidate_blocks = sorted(set(cs_block) | set(expert_block))
SEEDS = range(1, 6)
test_blocks_by_seed = {}
for seed in SEEDS:
    rng = np.random.default_rng(seed)
    test_blocks_by_seed[seed] = set(
        rng.choice(candidate_blocks, size=max(1, round(0.2 * len(candidate_blocks))), replace=False)
    )

print(f"{len(candidate_blocks)} candidate blocks (citizen-science ∪ expert presences)")
for seed, tb in test_blocks_by_seed.items():
    print(f"seed {seed}: {len(tb)} test blocks")


## 4. Run One-Class SVM, Random Forest and Logistic Regression across every level and seed

Same configurations as every earlier RQ1 notebook — `nu=0.05` for the
one-class SVM, `n_estimators=300`/`class_weight="balanced"` for Random
Forest, `class_weight="balanced"`/`max_iter=1000` for Logistic
Regression; no re-tuning here. For each seed, the test presence pool
(citizen-science test + expert test, regardless of mixing level) and
the background test/train split are fixed once; only the *training*
presence set changes across levels, per Section 2's effort order.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

from src.evaluation import continuous_boyce

pb_configs = [
    ("RandomForest", lambda: RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42)),
    ("LogisticRegression", lambda: LogisticRegression(class_weight="balanced", max_iter=1000)),
]

records = []
for seed in SEEDS:
    test_blocks = test_blocks_by_seed[seed]
    cs_test_mask = np.isin(cs_block, list(test_blocks))
    expert_test_mask = np.isin(expert_block, list(test_blocks))
    bg_test_mask = np.isin(bg_block, list(test_blocks))

    cs_train_emb, cs_test_emb = cs_emb[~cs_test_mask], cs_emb[cs_test_mask]
    expert_train_emb, expert_test_emb = expert_emb[~expert_test_mask], expert_emb[expert_test_mask]
    expert_train_poly = expert_poly_id[~expert_test_mask]
    bg_train_emb, bg_test_emb = bg_emb[~bg_test_mask], bg_emb[bg_test_mask]

    test_pres_emb = np.concatenate([cs_test_emb, expert_test_emb])
    if len(test_pres_emb) < 5 or len(bg_test_emb) < 5:
        continue

    for lvl in LEVELS:
        if lvl >= 100:
            train_pres_emb = expert_train_emb
        else:
            incl = included_polygons(lvl)
            train_pres_emb = np.concatenate([cs_train_emb, expert_train_emb[np.isin(expert_train_poly, incl)]])
        if len(train_pres_emb) < 5:
            continue

        # One-Class SVM: trained on presences only, no background
        scaler_oc = StandardScaler().fit(train_pres_emb)
        oc_model = OneClassSVM(kernel="rbf", nu=0.05, gamma="scale").fit(scaler_oc.transform(train_pres_emb))
        score_pres = oc_model.decision_function(scaler_oc.transform(test_pres_emb))
        score_bg = oc_model.decision_function(scaler_oc.transform(bg_test_emb))
        y_true = np.concatenate([np.ones(len(score_pres)), np.zeros(len(score_bg))])
        y_score = np.concatenate([score_pres, score_bg])
        records.append({
            "seed": seed, "level_pct": lvl, "classifier": "OneClassSVM",
            "n_train_presence": len(train_pres_emb),
            "roc_auc": roc_auc_score(y_true, y_score),
            "boyce": continuous_boyce(score_pres, score_bg),
        })

        # Presence-vs-background: Random Forest and Logistic Regression
        X_train = np.concatenate([train_pres_emb, bg_train_emb])
        y_train = np.concatenate([np.ones(len(train_pres_emb)), np.zeros(len(bg_train_emb))])
        X_test = np.concatenate([test_pres_emb, bg_test_emb])
        y_test = np.concatenate([np.ones(len(test_pres_emb)), np.zeros(len(bg_test_emb))])
        scaler_pb = StandardScaler().fit(X_train)
        X_train_s, X_test_s = scaler_pb.transform(X_train), scaler_pb.transform(X_test)

        for name, ctor in pb_configs:
            model = ctor().fit(X_train_s, y_train)
            score = model.predict_proba(X_test_s)[:, 1]
            sp, sb = score[y_test == 1], score[y_test == 0]
            records.append({
                "seed": seed, "level_pct": lvl, "classifier": name,
                "n_train_presence": len(train_pres_emb),
                "roc_auc": roc_auc_score(y_test, score),
                "boyce": continuous_boyce(sp, sb),
            })

results_df = pd.DataFrame(records)
print(f"{len(results_df)} (seed, level, classifier) rows; {results_df['seed'].nunique()} valid seeds")
results_df.head()


## 5. Aggregate across seeds


In [ ]:
summary = (
    results_df.groupby(["classifier", "level_pct"])
    .agg(roc_auc_mean=("roc_auc", "mean"), roc_auc_sd=("roc_auc", "std"),
         boyce_mean=("boyce", "mean"), boyce_sd=("boyce", "std"),
         n_seeds=("seed", "nunique"))
    .reset_index()
    .merge(nominal_df[["level_pct", "total_N"]], on="level_pct")
    .sort_values(["classifier", "level_pct"])
    .reset_index(drop=True)
)
summary.round(3)


## 6. Plot: performance vs. effort

Two panels (ROC-AUC, Boyce), not one dual-axis chart — the two metrics
are on different scales and a shared axis would misrepresent both.
One line per classifier, fixed categorical colours (blue = One-Class
SVM, orange = Random Forest, aqua = Logistic Regression), Random
Forest and Logistic Regression's background pool and test set
identical, so differences are only about mixing level and classifier.


In [ ]:
import matplotlib.pyplot as plt

COLORS = {"OneClassSVM": "#2a78d6", "RandomForest": "#eb6834", "LogisticRegression": "#1baf7a"}
MARKERS = {"OneClassSVM": "o", "RandomForest": "s", "LogisticRegression": "^"}

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharex=True)

for metric, ax, title in [("roc_auc", axes[0], "ROC-AUC"), ("boyce", axes[1], "Continuous Boyce index")]:
    for clf, color in COLORS.items():
        sub = summary[summary["classifier"] == clf].sort_values("level_pct")
        ax.plot(sub["level_pct"], sub[f"{metric}_mean"], color=color, marker=MARKERS[clf],
                markersize=6, linewidth=2, label=clf)
        ax.fill_between(sub["level_pct"], sub[f"{metric}_mean"] - sub[f"{metric}_sd"],
                         sub[f"{metric}_mean"] + sub[f"{metric}_sd"], color=color, alpha=0.12)
    ax.set_title(title)
    ax.set_xlabel("% of expert polygons added to citizen-science base")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#e5e4e0", linewidth=0.8, zorder=0)

axes[0].set_ylabel("score (mean ± sd over 5 seeds)")
axes[0].legend(frameon=False, loc="lower right")
fig.suptitle("RQ1 effort-reduction curve — TESSERA embeddings", y=1.02)
fig.tight_layout()
plt.show()


## 7. Reading the curve

**Random Forest and Logistic Regression are essentially flat from 0%
to 90% expert, then drop at 100%.** Random Forest: ROC-AUC 0.879 (0%)
→ 0.882 (20%) → 0.878 (90%) → 0.801 (100%); Boyce 0.993 (0%) → 0.994
(20%) → 0.983 (90%) → 0.917 (100%) — every 10–90% point is within one
seed's noise of the 0% point. Logistic Regression shows a small,
genuine rise (ROC-AUC 0.873 → 0.896 across 0–90%, just outside the sd
band) before the same drop at 100%. **Under this evaluation, adding
expert data in the 10–90% range buys presence-vs-background models
next to nothing — the practical gain only shows up if expert data
entirely replaces citizen science (100%), and that replacement makes
things *worse*, not better**, because the test set always includes
citizen-science presences the expert-only model never saw (see point 4
in "Design decisions").

**A likely reason, visible in the numbers themselves:** at every
10–90% level, citizen science still supplies 85–100% of the training
presences (e.g. at 90%, 2 995 citizen-science vs. 2 504 expert pixels,
total_N 5 499 but expert is still under half) — the citizen-science
signal dominates training throughout, so a model already near ceiling
on it (Boyce ≈0.98–0.99 from 0%) has little room left for the growing
expert minority to move the needle. This also fits Dimson et al.
(2023)'s point that the two sources have different biases: only
removing citizen science outright (100%) changes what the model is
being told, and here that change hurts generalisation back to
citizen-science-style presences rather than helping.

**One-Class SVM is the exception: it genuinely improves with more
expert effort.** ROC-AUC rises monotonically, 0.574 (0%) → 0.655 (70%)
→ 0.765 (100%) — a real trend, not noise, though it remains well below
both presence-vs-background classifiers throughout, and its Boyce
index is noisy (0.63–0.77 range, sd up to 0.30) with no clean trend.
This is consistent with a true one-class model depending entirely on
how well its *training* presences describe the class boundary — more,
less citizen-science-biased data helps it more directly than it helps
a model already using background to separate the classes.

**These numbers are not directly comparable to the earlier standalone
notebooks' 0% and 100% results**, because this notebook deliberately
changed what "test set" means (point 4 above): `one_class_method_selection.ipynb`
and `presence_background_citizen_science.ipynb` (0%) drew test blocks
from the full 85-block pilot extent; `expert_arm_classifiers.ipynb`
(100%) tested only against expert presences. Here, every level is
tested against the same combined citizen-science + expert pool drawn
from only the 47 blocks that contain either source. Lower scores here
(e.g. 100% Random Forest: 0.801/0.917 here vs. 0.990/0.954 there) are
the direct, informative consequence of also requiring generalisation
to citizen-science presences, not a contradiction.

**Tentative answer to "where's the balance point" (first pass, many
caveats below):** for the classifiers tested, **no amount of expert
data short of full replacement earns a visible improvement** — the
practical implication, if this holds up, is that the current RQ1
setup does not justify the field-collection cost, since citizen
science alone already captures effectively all of the presence-vs-
background performance achievable here. This is the opposite of a
"more expert data is better" story and deserves scrutiny, not
automatic acceptance, before treating it as a conclusion.

**Caveats (first pass, not a definitive benchmark):**

- Only one background-pool design and one set of hyperparameters are
  used throughout — none of this is re-tuned per mixing level, and the
  background pool is still not confirmed by the supervisor.
- The flat region could be a ceiling effect (Boyce already ≈0.98–0.99
  for Random Forest at 0%) rather than genuine evidence that expert
  data is worthless — a harder test set or a different metric might
  reveal differences the current one cannot.
- Effort is measured as % of expert *polygons*, not weighted by
  polygon area or by true field-collection cost (travel time, access
  difficulty) — a simplification.
- Compact neural network candidate still not tested.
- Only TESSERA; Sentinel-2 is deferred (per the adopted RQ1 design).
- Test-block sampling draws from only 47 candidate blocks (citizen
  science ∪ expert presence blocks), a different and smaller universe
  than the 85-block pilot extent used in earlier standalone notebooks
  — see "not directly comparable" above.

**Onde/Where:** full reasoning and the three open design decisions in
`thesis/notes.md`; status in `docs/research_questions.md`;
`docs/todo.md` for what follows.
